In [ ]:
import harmonypy as hm
import numpy as np
import scanpy as sc

In [ ]:
adata1 = sc.read_h5ad("STH_singlet.h5ad")
adata2 = sc.read_h5ad("NAC.h5ad")

In [ ]:
adata1.obs["batch"] = "dataset1"
adata2.obs["batch"] = "dataset2"



In [ ]:
adata = sc.concat([adata1, adata2], join="inner", label="batch_source", index_unique="-")

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key="batch")


In [ ]:
sc.pp.pca(adata, n_comps=50, mask_var="highly_variable")


In [ ]:
ho = hm.run_harmony(adata.obsm["X_pca"].astype(np.float64), adata.obs, ["batch"])
adata.obsm["X_pca_harmony"] = ho.Z_corr if ho.Z_corr.shape[0] == adata.n_obs else ho.Z_corr.T

sc.pp.neighbors(adata, use_rep="X_pca_harmony")
sc.tl.umap(adata)
sc.tl.leiden(adata)

In [ ]:
sc.pl.umap(adata, color=['roi', 'cell_type'], legend_loc="on data")

In [ ]:
adata